# Imports

In [1]:
import os
import glob
import sqlite3
import pandas as pd

# Inputs

In [2]:
DB_PATH   = "../data/sqlite/data.db"
DATA_ROOT = "../../../Datasets"

# Dataset folder names to include (e.g. ["UNSW_NB15", "Aposemat_IoT_23"])
# Leave empty to process all datasets found under DATA_ROOT
INCLUDED_DATASETS = ["benign_traffic"]
TABLE_NAME = "network_data"

# Load dataset CSVs into database

In [3]:
def load_dataset_to_db(data_root: str, dataset: str, db_path: str, chunksize: int = 50_000) -> None:
    dataset_path = os.path.join(data_root, dataset)

    csv_paths = sorted(
        glob.glob(os.path.join(dataset_path, "**", "merged_*.csv"), recursive=True)
    )

    if not csv_paths:
        print(f"[{dataset}] No merged CSVs found — skipping.")
        return

    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)

    print(f"[{dataset}] {len(csv_paths)} CSV file(s) → table '{TABLE_NAME}'")

    for csv_path in csv_paths:
        print(f"  {os.path.basename(csv_path)} ...", end=" ", flush=True)
        for chunk in pd.read_csv(csv_path, chunksize=chunksize):
            chunk.to_sql(TABLE_NAME, conn, if_exists="append", index=False)
        conn.commit()
        print("✓")

    conn.execute(f'CREATE INDEX IF NOT EXISTS "idx_{TABLE_NAME}_label" ON "{TABLE_NAME}" (label)')
    conn.commit()

    total = conn.execute(f'SELECT COUNT(*) FROM "{TABLE_NAME}"').fetchone()[0]
    print(f"\n✓ Table '{TABLE_NAME}': {total:,} rows")
    conn.close()

In [4]:
datasets = [
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
]

In [5]:
all_datasets = [
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
]

if INCLUDED_DATASETS:
    datasets = [d for d in all_datasets if d in set(INCLUDED_DATASETS)]
else:
    datasets = all_datasets

In [6]:
for dataset in datasets:
    load_dataset_to_db(DATA_ROOT, dataset, DB_PATH)

[benign_traffic] 1 CSV file(s) → table 'network_data'
  merged_benign.csv ... ✓

✓ Table 'network_data': 883,774,201 rows
